# 缺失值处理：检测、填充与删除的完整策略
> 难度标签：初级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：01_数据预处理 | 源文件：缺失值处理.py | 核心功能：7种缺失值处理策略的对比演示

## 概述

缺失值是真实数据中最常见的"脏数据"类型。数据库导出异常、问卷漏填、传感器故障……都可能产生缺失值。大多数机器学习算法**无法处理缺失值**（XGBoost 等少数除外），所以缺失值处理是数据预处理的第一道关卡。

这个脚本演示了从最简单的"删掉它"到最复杂的"多重插补"共 5 种填充策略和 2 种删除策略，并在最后对比了各策略对数据分布的影响。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

## 数学原理

### 1. 缺失机制的统计分类

缺失值处理的第一步是理解**缺失机制**（Missing Mechanism），它决定了填充策略的有效性：

**MCAR（Missing Completely At Random）**：

$$P(M_i = 1 | X, Y) = P(M_i = 1) = \pi$$

缺失概率与任何变量都无关。例如：传感器随机故障。这是最强的假设——删除缺失样本不会引入偏差。

**MAR（Missing At Random）**：

$$P(M_i = 1 | X, Y) = P(M_i = 1 | X_{\text{obs}})$$

缺失概率只依赖于**已观测到**的变量。例如：年轻人更不愿意填收入（年龄已观测到）。条件于观测变量后，缺失是随机的。

**MNAR（Missing Not At Random）**：

$$P(M_i = 1 | X, Y) = P(M_i = 1 | X, Y, X_{\text{miss}})$$

缺失概率依赖于**缺失值本身**。例如：高收入者不愿透露收入。这是最难处理的情况——简单删除和填充都会引入偏差。

| 缺失机制 | 删除法有偏？ | 简单填充有偏？ | 需要建模？ |
|----------|------------|--------------|----------|
| MCAR | 否 | 否 | 否 |
| MAR | 是 | 部分 | 是 |
| MNAR | 是 | 是 | 是（且困难） |

### 2. 均值填充的方差压缩效应

**代码对应**：`SimpleImputer(strategy="mean")` 用均值填充。

设特征 $j$ 的真实方差为 $\sigma_j^2$，缺失率为 $p$。均值填充后，$n_{\text{obs}} = (1-p)n$ 个观测值保持不变，$n_{\text{miss}} = pn$ 个缺失值被替换为均值 $\bar{x}$。

填充后的方差为：

$$\hat{\sigma}_{j,\text{filled}}^2 = \frac{1}{n}\left[\sum_{i \in \text{obs}} (x_{ij} - \bar{x}_j)^2 + \underbrace{\sum_{i \in \text{miss}} (\bar{x}_j - \bar{x}_j)^2}_{= 0}\right] = \frac{n_{\text{obs}}}{n} \hat{\sigma}_{j,\text{obs}}^2 = (1-p) \hat{\sigma}_{j,\text{obs}}^2$$

即方差被压缩为原来的 $(1-p)$ 倍。缺失率 20% 时方差损失 20%，这会影响所有依赖方差的后续分析（如 PCA、假设检验）。

**中位数填充**对偏态分布更鲁棒，但也存在类似的方差压缩问题。

### 3. KNN 填充的距离加权公式

**代码对应**：`KNNImputer(n_neighbors=5, weights="distance")` 使用距离加权的 K 近邻。

对于含缺失值的样本 $\mathbf{x}$，设缺失特征为 $j$，KNN 填充过程为：

1. 在非缺失特征空间中计算 $\mathbf{x}$ 到所有样本的距离：

$$d(\mathbf{x}, \mathbf{x}_i) = \sqrt{\sum_{j' \in \text{obs}(\mathbf{x})} (x_{j'} - x_{ij'})^2}$$

注意：距离计算**只使用非缺失特征**。

2. 选取距离最近的 $K$ 个邻居 $\mathcal{N}_K(\mathbf{x})$

3. 填充值为加权平均：

$$\hat{x}_j = \frac{\sum_{i \in \mathcal{N}_K(\mathbf{x})} w_i \cdot x_{ij}}{\sum_{i \in \mathcal{N}_K(\mathbf{x})} w_i}$$

其中 $w_i$ 为权重：
- `weights="uniform"`：$w_i = 1$（等权平均）
- `weights="distance"`：$w_i = 1/d(\mathbf{x}, \mathbf{x}_i)$（距离越近权重越大）

**距离加权的优势**：如果最近邻距离为 0.1，次近邻距离为 10，等权平均会让远处的邻居"拉偏"填充值。距离加权有效抑制了这一问题。

**计算复杂度**：朴素 KNN 需要 $O(n^2 p)$ 次距离计算。对大数据集可使用 KD-Tree（$p < 20$ 时有效）或 Ball Tree 加速。

### 4. 迭代插补（MICE）的联合分布建模

**代码对应**：`IterativeImputer(max_iter=10)` 实现了 MICE 算法。

MICE（Multiple Imputation by Chained Equations）基于**全条件分布**（Full Conditional Specifications）的思想：不直接建模联合分布 $P(X_1, X_2, \ldots, X_p)$，而是对每个变量 $X_j$ 的条件分布 $P(X_j | X_{-j})$ 分别建模。

**算法流程**：

1. **初始化**：对所有缺失值用简单方法（如均值）填充，得到完整数据集 $\mathbf{X}^{(0)}$

2. **迭代**：对 $t = 1, 2, \ldots, T$，依次更新每个含缺失值的特征 $j$：

   a. 用当前数据 $\mathbf{X}^{(t-1)}$ 中的 $\mathbf{X}_{-j}$ 作为自变量，$\mathbf{X}_j$ 作为因变量，拟合回归模型 $\hat{f}_j$

   b. 对缺失样本 $i$，用 $\hat{f}_j$ 预测并更新：$x_{ij}^{(t)} \sim P(X_j | \mathbf{x}_{i,-j}^{(t-1)}, \hat{f}_j)$

3. 重复 $T$ 轮后收敛到多元分布的平稳近似

sklearn 的 `IterativeImputer` 默认使用**贝叶斯岭回归**作为每个条件分布的模型：

$$P(X_j | X_{-j}) \sim \mathcal{N}(\mathbf{X}_{-j} \hat{\mathbf{w}}_j, \hat{\sigma}_j^2)$$

贝叶斯岭回归自动给系数加 L2 正则化，避免了条件模型的过拟合。

**收敛性**：在 MAR 假设下，MICE 算法在迭代次数 $T \to \infty$ 时收敛到目标联合分布的唯一不动点（前提是各条件分布兼容——即存在对应的联合分布）。

### 5. 各方法的计算复杂度对比

| 方法 | 时间复杂度 | 空间复杂度 | 适用规模 |
|------|-----------|-----------|---------|
| 均值/中位数填充 | $O(np)$ | $O(p)$ | 任意 |
| KNN 填充 | $O(n^2 p)$ | $O(n^2)$ | $n < 10^4$ |
| MICE（$T$ 轮迭代） | $O(T \cdot n p^2)$ | $O(np)$ | 中等 |

### 6. 删除策略的偏差分析

**代码对应**：`data.dropna()` 删除所有含缺失值的行。

**完全删除（listwise deletion）**的偏差取决于缺失机制：

- **MCAR**：删除后样本仍然是总体的随机子集，**无偏**
- **MAR**：删除后样本不再随机，**有偏**
- **MNAR**：删除后样本有系统性偏差，**有偏**

删除后有效样本量为 $n_{\text{eff}} = n \cdot (1 - p_{\text{any}})$，其中 $p_{\text{any}}$ 为至少有一个特征缺失的样本比例。当特征数 $p$ 很大时：

$$p_{\text{any}} = 1 - \prod_{j=1}^{p}(1 - p_j)$$

即使每个特征的缺失率都很低（如 $p_j = 2\%$），当 $p = 50$ 时，$p_{\text{any}} = 1 - 0.98^{50} \approx 64\%$——删除后仅剩 36% 的数据。这就是为什么在高维数据中，**删除策略几乎不可用**。

### 7. 缺失值指示器的信息提取

对于特征 $j$，创建二值指示变量：

$$r_{ij} = \begin{cases} 1 & \text{if } x_{ij} \text{ 缺失} \\ 0 & \text{otherwise} \end{cases}$$

在 MAR 机制下，缺失模式本身可能包含预测信息。将 $r_{ij}$ 作为额外特征加入模型，可以捕获这种信息。

数学上，这等价于建模：

$$P(Y | X_{\text{obs}}, M) = P(Y | X_{\text{obs}}) + \boldsymbol{\beta}^T \mathbf{M}$$

其中 $\mathbf{M} = (r_1, \ldots, r_p)$ 为缺失指示向量。如果 $\boldsymbol{\beta} \neq \mathbf{0}$，说明缺失模式与目标变量相关。

## 代码结构

| 段落 | 内容 | 方法 |
|------|------|------|
| 第1节 | 构造含缺失值的 200 行示例数据 | 随机注入约 10% 缺失值 |
| 第2节 | 删除策略 | dropna() 和阈值删除 |
| 第3节 | 统计量填充 | 均值、中位数、众数 |
| 第4节 | 常量填充 | 用 0 填充 |
| 第5节 | KNN 填充 | 基于最近邻加权平均 |
| 第6节 | 迭代插补 | 模拟 R 的 MICE 算法 |
| 第7节 | 对比各策略的均值差异 | — |

### 1. 构造含缺失值的示例数据

运行 1. 构造含缺失值的示例数据 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
n = 200
data = pd.DataFrame({
    "年龄": np.random.randint(18, 70, n).astype(float),
    "收入": np.random.normal(50000, 15000, n),
# --- 数值计算 ---
    "学历": np.random.choice(["高中", "本科", "硕士", "博士"], n),
    "信用分": np.random.uniform(300, 850, n),
})

# 随机注入缺失值（约 10%）
for col in ["年龄", "收入", "信用分"]:
    mask = np.random.rand(n) < 0.10
    data.loc[mask, col] = np.nan

# 学历也注入少量缺失
mask_edu = np.random.rand(n) < 0.05
data.loc[mask_edu, "学历"] = np.nan

print("=== 原始数据（前 10 行）===")
print(data.head(10))
print(f"\n各列缺失数量:\n{data.isnull().sum()}")
print(f"总缺失率: {data.isnull().mean().mean():.2%}")

### 2. 删除策略

运行 2. 删除策略 的代码，观察算法在该环节的行为。

In [ ]:
# 2a. 删除含缺失值的行（dropna）
data_dropped = data.dropna()
print(f"\n=== dropna 后剩余行数: {len(data_dropped)} (原始 {len(data)}) ===")

# 2b. 设定阈值：删除缺失超过 30% 列的行
thresh = int(data.shape[1] * 0.7)  # 至少 70% 非空
data_thresh = data.dropna(thresh=thresh)
print(f"thresh={thresh} 后剩余行数: {len(data_thresh)}")

### 3. 均值/中位数/众数填充

运行 3. 均值/中位数/众数填充 的代码，观察算法在该环节的行为。

In [ ]:
data_num = data[["年龄", "收入", "信用分"]]

# 均值填充
imputer_mean = SimpleImputer(strategy="mean")
filled_mean = pd.DataFrame(imputer_mean.fit_transform(data_num), columns=data_num.columns)
print(f"\n=== 均值填充后缺失: {filled_mean.isnull().sum().sum()} ===")

# 中位数填充
imputer_median = SimpleImputer(strategy="median")
filled_median = pd.DataFrame(imputer_median.fit_transform(data_num), columns=data_num.columns)
print(f"中位数填充后缺失: {filled_median.isnull().sum().sum()}")

# 众数填充（适用于分类特征）
imputer_mode = SimpleImputer(strategy="most_frequent")
edu_filled = imputer_mode.fit_transform(data[["学历"]])
print(f"学历众数填充后缺失: {pd.DataFrame(edu_filled).isnull().sum().sum()}")

### 4. 常量填充

运行 4. 常量填充 的代码，观察算法在该环节的行为。

In [ ]:
imputer_const = SimpleImputer(strategy="constant", fill_value=0)
filled_const = pd.DataFrame(imputer_const.fit_transform(data_num), columns=data_num.columns)
print(f"\n=== 常量(0)填充后缺失: {filled_const.isnull().sum().sum()} ===")

### 5. KNN 填充

运行 5. KNN 填充 的代码，观察算法在该环节的行为。

In [ ]:
# 利用 K 个最近邻样本的加权平均值填充，考虑特征间相关性
imputer_knn = KNNImputer(n_neighbors=5, weights="distance")
filled_knn = pd.DataFrame(imputer_knn.fit_transform(data_num), columns=data_num.columns)
print(f"\n=== KNN 填充后缺失: {filled_knn.isnull().sum().sum()} ===")
print("KNN 填充后的统计描述:")
print(filled_knn.describe().round(2))

### 6. 多重插补（IterativeImputer）

运行 6. 多重插补（IterativeImputer） 的代码，观察算法在该环节的行为。

In [ ]:
# 模拟 R 的 MICE：对每个特征建立回归模型，迭代填充
imputer_iter = IterativeImputer(max_iter=10, random_state=42)
filled_iter = pd.DataFrame(imputer_iter.fit_transform(data_num), columns=data_num.columns)
print(f"\n=== 迭代插补后缺失: {filled_iter.isnull().sum().sum()} ===")
print("迭代插补后统计描述:")
print(filled_iter.describe().round(2))

### 7. 对比各策略的填充效果

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== 各策略填充后的均值对比 ===")
original_mean = data_num.mean()
print(f"原始均值:\n{original_mean.round(2)}\n")
for name, filled in [("均值", filled_mean), ("中位数", filled_median),
                      ("常量(0)", filled_const), ("KNN", filled_knn), ("迭代", filled_iter)]:
# --- 输出结果 ---
    print(f"{name}填充均值:\n{filled.mean().round(2)}\n")

## 关键代码解释

### 删除策略 —— 最简单但最浪费

```python
data_dropped = data.dropna()              # 任何列有缺失就删行
data_thresh = data.dropna(thresh=thresh)  # 缺失超过阈值才删
```

dropna() 会删除**任何列**包含缺失值的行。如果数据有 20 列，其中 1 列缺失率 50%，用 dropna() 可能丢失一半数据。	hresh 参数更实用——只有当非空列数低于阈值时才删除，保留了"大部分完整"的行。

**何时使用**：缺失率极低（< 5%），或者数据量远大于需求时。

### 统计量填充 —— 经典但有盲点

```python
imputer_mean = SimpleImputer(strategy="mean")
imputer_median = SimpleImputer(strategy="median")
imputer_mode = SimpleImputer(strategy="most_frequent")
```

SimpleImputer 是 sklearn 的标准填充器。均值填充对正态分布数据效果好，但对偏态分布（如收入）会被极端值拉偏。中位数更鲁棒。众数用于分类特征。

**陷阱**：均值/中位数填充会**压缩方差**——所有缺失值都被同一个数替代，数据的变异性被人为降低了。

### KNN 填充 —— 利用相似样本

```python
imputer_knn = KNNImputer(n_neighbors=5, weights="distance")
```

对每个含缺失值的样本，找到 K 个"最近邻"（基于非缺失特征），用它们的加权平均值填充。weights="distance" 让更近的邻居有更大话语权。

**优势**：考虑了特征间的相关性。**劣势**：计算复杂度 O(n²)，大数据集会很慢。需要先做特征缩放。

### 迭代插补（MICE）—— 最精确但最慢

```python
imputer_iter = IterativeImputer(max_iter=10, random_state=42)
```

模拟 R 语言的 MICE（Multiple Imputation by Chained Equations）算法。核心思想：对每个含缺失值的特征，用其他特征作为预测变量建立回归模型，然后迭代更新。重复 max_iter 轮后收敛。

**这是理论上最好的单一填充方法**，但计算量大，且对回归模型的选择敏感（默认是贝叶斯岭回归）。

## 使用示例

```python
from sklearn.impute import SimpleImputer, KNNImputer

# 快速均值填充
imputer = SimpleImputer(strategy="mean")
X_filled = imputer.fit_transform(X_train)
X_test_filled = imputer.transform(X_test)  # 用训练集的均值填充测试集
```

## 注意事项

1. **数据泄露**：填充器必须在训练集上 
it，然后用同一个填充器 	ransform 测试集
2. **缺失机制很重要**：MCAR（完全随机缺失）、MAR（随机缺失）、MNAR（非随机缺失）需要不同的处理策略
3. **分类特征**：KNN 和迭代插补对分类特征需要先编码，或者单独用众数填充
4. **方差压缩**：均值/中位数填充会人为降低方差，后续分析中需要注意这一点
5. **性能权衡**：简单填充快但粗糙，KNN 和 MICE 精确但慢，应根据数据规模和业务需求选择

## 总结与延伸

以上代码展示了 **缺失值处理** 的完整流程。

**解读要点**：
- 观察处理前后数据的 **统计分布变化**
- 关注缺失值填充策略对下游模型的影响
- 对比不同缩放方法的适用场景

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **多重插补**（Multiple Imputation）：不是填充一次，而是填充多次，得到多个完整数据集，分别建模后合并结果，能更真实地反映缺失值带来的不确定性
- **缺失值本身是特征**：缺失的模式可能包含信息（比如"收入"缺失的人可能不愿意透露），可以创建一个"是否缺失"的二值特征
- **XGBoost 的原生支持**：XGBoost 等树模型能自动处理缺失值，把它当作一个独立的分支方向
